# Exploration — transitlens-data-pipeline

Loads all three synthetic candidates through `interface.load_light_curve()` (the same entry point `transitlens-ml-core` uses) and inspects them visually.

This notebook is exploratory only — nothing here is imported by the pipeline itself. See `CONTRIBUTING.md`'s "boundary rule" for why plotting code lives here and not in the library.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np

from interface import load_light_curve

## Load all three candidates

`config={'generate': True}` means this cell works even on a brand-new checkout with no pre-generated CSVs yet.

In [ ]:
target_ids = ["candidate_a", "candidate_b", "candidate_c"]
results = {
    tid: load_light_curve("synthetic", tid, config={"generate": True})
    for tid in target_ids
}

for tid, r in results.items():
    m = r["metadata"]
    print(f"{tid:12s} n_points={r['n_points']:6d}  label={m['label']:22s} "
          f"true_period={m['true_period']}  true_depth={m['true_depth']}")

## Full light curves

Each candidate over the full 27-day observation window.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

for ax, tid in zip(axes, target_ids):
    r = results[tid]
    ax.plot(r["time"], r["flux"], ".", markersize=1, color="steelblue")
    ax.set_ylabel("flux")
    ax.set_title(f"{tid} — {r['metadata']['label']}")
    ax.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")

axes[-1].set_xlabel("time (BTJD)")
fig.tight_layout()
plt.show()

## Phase-folded view of candidate_a

Folding on the known `true_period` collapses every transit onto a single profile — this is the same operation `ml-core`'s BLS detector performs internally, just done here with the *known* period for a sanity check rather than searching for it.

In [ ]:
r = results["candidate_a"]
time = np.array(r["time"])
flux = np.array(r["flux"])
period = r["metadata"]["true_period"]

phase = ((time % period) / period)
phase = np.where(phase > 0.5, phase - 1.0, phase)

order = np.argsort(phase)

plt.figure(figsize=(9, 4))
plt.plot(phase[order], flux[order], ".", markersize=2, color="darkorange")
plt.xlabel("phase")
plt.ylabel("flux")
plt.title(f"candidate_a phase-folded on true_period={period} days")
plt.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")
plt.tight_layout()
plt.show()

## Error-path sanity checks

Quick reminder of what `load_light_curve()` does on bad input — useful when debugging an `ml-core` integration issue.

In [ ]:
for source, target_id in [("unknown_source", "x"), ("synthetic", "nonexistent")]:
    try:
        load_light_curve(source, target_id)
    except Exception as e:
        print(f"{source!r}, {target_id!r} -> {type(e).__name__}: {e}")